In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.stats.multitest import multipletests

from study.data import cohort, features, prediction_matrix, prediction_metadata
from study.features import DYNAMIC_VARS
from study.io import WORK, load, write_table
from study.models import fit_source_model, predict
from study.statistics import paired_difference, prediction_rows, risk_matrix

In [ ]:
GLOBAL_FEATURES = (
    'total_measurements',
    'total_measurement_change',
    'distinct_variables_measured',
    'distinct_variables_change'
)
COMMON_GLOBALS = {name: 'common__' + name for name in GLOBAL_FEATURES}

In [ ]:
source_fits = load(WORK / 'models/source.pkl')

sets = load(WORK / 'feature_sets.pkl')
source_metadata = cohort('metavision')

In [ ]:
def observation_summaries(frame, variables):
    variables = list(variables)

    # Collect first day, second day, and total observation counts
    first = frame[[name + '_count_d1' for name in variables]].to_numpy(float)
    second = frame[[name + '_count_d2' for name in variables]].to_numpy(float)
    total = frame[[name + '_count' for name in variables]].to_numpy(float)
    values = np.concatenate([first, second, total], axis = 1)

    # Summarise total measurements and distinct variables
    return pd.DataFrame(
        {
            'total_measurements': total.sum(axis = 1),
            'total_measurement_change': (second - first).sum(axis = 1),
            'distinct_variables_measured': (total > 0).sum(axis = 1),
            'distinct_variables_change': (second > 0).sum(axis = 1) - (first > 0).sum(axis = 1)
        },
        index = frame.index,
    )

In [ ]:
# Rebuild global observation summaries without ventilation variables
matrices = {}
for domain in ('metavision', 'mimiciv', 'eicu'):
    x = features(domain).copy()
    common = observation_summaries(x, [variable for variable in DYNAMIC_VARS if variable != 'MechVent'])

    for name in GLOBAL_FEATURES:
        x[COMMON_GLOBALS[name]] = common[name]

    matrices[domain] = x


In [ ]:
# Restore the global summaries while excluding ventilation-specific features
restored = [
    COMMON_GLOBALS.get(column, column) for column in sets['original']['R3'] if not column.startswith('MechVent_')
]

# Refit R3 using the restored feature set
rows = []
predictions = []
for repeat in source_fits['repeats']:
    fit_ids = source_fits['fit_ids']['metavision', repeat]
    fitted = fit_source_model(
        matrices['metavision'].loc[fit_ids, restored],
        source_metadata.loc[fit_ids, 'in_hospital_mortality'],
        'xgboost',
        source_fits['parameters'],
    )

    # Pair restored R3 predictions with the existing shared R2 model
    for domain in ('metavision', 'mimiciv', 'eicu'):
        metadata = cohort(domain).sort_index()
        ids = metadata.index[metadata.partition.eq('test')] if domain == 'eicu' else metadata.index

        for tier, bundle in [('R3', fitted), ('R2', source_fits['models']['metavision', repeat, 'shared', 'R2'])]:
            predictions.append(
                prediction_rows(
                    prediction_metadata(metadata.loc[ids]),
                    bundle['oof'].loc[ids].to_numpy()
                    if domain == 'metavision'
                    else predict(bundle['model'], matrices[domain].loc[ids]),
                    repeat = repeat,
                    domain = domain,
                    tier = tier,
                )
            )

predictions = pd.concat(predictions, ignore_index = True)

In [ ]:
# Compare restored R3 against R2 and ordinary shared R3
for domain, frame in predictions.groupby('domain'):
    ids = pd.Index(sorted(frame.stay_id.unique()))
    candidate = risk_matrix(frame.loc[frame.tier.eq('R3')], ids)
    comparisons = [('R3_minus_R2', risk_matrix(frame.loc[frame.tier.eq('R2')], ids))]

    ordinary = prediction_matrix(
        'full',
        ids,
        [
            ('source', '==', 'metavision'),
            ('domain', '==', domain),
            ('support', '==', 'shared'),
            ('tier', '==', 'R3')
        ],
    )

    comparisons.append(('restored_R3_minus_shared_R3', ordinary))

    for contrast, reference in comparisons:
        for equal_hospital in [False, True] if domain == 'eicu' else [False]:
            rows.append(
                dict(
                    domain = domain,
                    support = 'restored',
                    contrast = contrast,
                    **paired_difference(
                        prediction_metadata(cohort(domain).loc[ids]),
                        candidate,
                        reference,
                        cluster = domain == 'eicu',
                        equal_hospital = equal_hospital,
                    ),
                )
            )

results = pd.DataFrame(rows)

# Adjust p values within each target and weighting family
for _, family in results.groupby(['domain', 'weighting']):
    results.loc[family.index, 'holm_p'] = multipletests(family.p_value, method = 'holm')[1]

write_table(WORK / 'results/ventilation.parquet', results)

display(results.round(5))